# Part 6: Evaluation and Selection
## 6.1 Threshold/Output Calibration

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix, f1_score

print("Generating out-of-fold probabilities for threshold evaluation...")

model_for_calibration = random_search.best_estimator_

oof_probs = cross_val_predict(
    model_for_calibration,
    X,
    y,
    cv=cv_strategy,
    method="predict_proba",
    n_jobs=-1
)

# Fit once so we can safely get the class order
model_for_calibration.fit(X, y)

classes = model_for_calibration.named_steps["model"].classes_

print("Class order:")
print(classes)

class_to_index = {label: idx for idx, label in enumerate(classes)}

idx_1 = class_to_index[1]
idx_2 = class_to_index[2]
idx_3 = class_to_index[3]
idx_4 = class_to_index[4]
idx_5 = class_to_index[5]

def apply_custom_threshold(probs, threshold_12):
    preds = []

    for p in probs:
        prob_12 = p[idx_1] + p[idx_2]

        if prob_12 >= threshold_12:
            if p[idx_1] >= p[idx_2]:
                preds.append(1)
            else:
                preds.append(2)
        else:
            remaining_indices = [idx_3, idx_4, idx_5]
            best_remaining_index = remaining_indices[np.argmax(p[remaining_indices])]
            preds.append(classes[best_remaining_index])

    return np.array(preds)

thresholds = [0.50, 0.40, 0.30, 0.25, 0.20]

results = []

true_dissatisfied = y.isin([1, 2])

for t in thresholds:
    custom_preds = apply_custom_threshold(oof_probs, t)

    pred_dissatisfied = np.isin(custom_preds, [1, 2])

    tp = np.sum(true_dissatisfied & pred_dissatisfied)
    fp = np.sum((~true_dissatisfied) & pred_dissatisfied)
    fn = np.sum(true_dissatisfied & (~pred_dissatisfied))

    recall_12 = tp / (tp + fn) if (tp + fn) > 0 else 0
    precision_12 = tp / (tp + fp) if (tp + fp) > 0 else 0

    weighted_f1 = f1_score(y, custom_preds, average="weighted")
    macro_f1 = f1_score(y, custom_preds, average="macro")

    results.append({
        "Threshold P(1 or 2)": t,
        "Weighted F1": weighted_f1,
        "Macro F1": macro_f1,
        "Recall for Ratings 1-2": recall_12,
        "Precision for Ratings 1-2": precision_12,
        "False Alarms": fp,
        "Missed Interventions": fn
    })

calibration_df = pd.DataFrame(results)

print("\nThreshold Trade-off Table for Dissatisfied Customers")
print("Lower thresholds catch more dissatisfied customers but usually create more false alarms.")

display(calibration_df.round(4))

chosen_threshold = calibration_df.sort_values(
    by="Weighted F1",
    ascending=False
).iloc[0]["Threshold P(1 or 2)"]

final_preds = apply_custom_threshold(oof_probs, chosen_threshold)

print(f"\nSelected Threshold: {chosen_threshold}")

print("\nClassification Report with Selected Custom Threshold:")
print(classification_report(y, final_preds, zero_division=0))

print("\nConfusion Matrix with Selected Custom Threshold:")

cm = confusion_matrix(
    y,
    final_preds,
    labels=classes
)

cm_df = pd.DataFrame(
    cm,
    index=[f"True {i}" for i in classes],
    columns=[f"Pred {i}" for i in classes]
)

display(cm_df)

#### Threshold / Output Calibration Results and Interpretation


For the multi-class rating prediction problem, the default decision rule is to choose the class with the highest predicted probability. However, this may not always be the best business decision because some errors are more costly than others.

In this project, ratings 1 and 2 represent dissatisfied customers. Missing these customers is costly because the business may lose the opportunity to follow up, resolve the issue, or prevent customer churn. Because of this, I tested custom thresholds based on the combined predicted probability of ratings 1 and 2.

The threshold results show a clear tradeoff. At the selected threshold of 0.50, the model achieved the best weighted F1 score of 0.61 and macro F1 score of 0.60. This threshold had recall of 0.83 for dissatisfied customers and precision of 0.92. It also produced 36 false alarms and 84 missed interventions.

Lowering the threshold increased recall for dissatisfied customers. For example, at threshold 0.20, recall increased to 0.92, meaning the model caught more dissatisfied customers. However, precision dropped to 0.81 and false alarms increased to 109. This means the business would catch more unhappy customers but would also spend more effort following up with customers who were not actually dissatisfied.

The selected threshold of 0.50 is the best balanced choice because it produced the highest weighted F1 while still maintaining strong dissatisfied-customer recall. If the business wants to prioritize service recovery over efficiency, a lower threshold such as 0.30 or 0.25 could also be considered because it catches more dissatisfied customers, even though it creates more false alarms.

## 6.2 Ablation Analysis

In [ ]:
if 'ablation_df' in locals() and not ablation_df.empty:
    comparison_rows = []

    # Iterate through each model type evaluated in the ablation study
    for model_name in ablation_df['Model'].unique():
        model_subset = ablation_df[ablation_df['Model'] == model_name]

        # Extract scores for each configuration
        struct_row = model_subset[model_subset['Configuration'] == 'Structured-only']
        text_row = model_subset[model_subset['Configuration'] == 'Text-only']
        comb_row = model_subset[model_subset['Configuration'] == 'Combined']

        # Safely get values if they exist
        if not struct_row.empty and not text_row.empty and not comb_row.empty:
            struct_mean = struct_row['Weighted F1 Mean'].values[0]
            text_mean = text_row['Weighted F1 Mean'].values[0]
            comb_mean = comb_row['Weighted F1 Mean'].values[0]
            comb_ci = comb_row['Weighted F1 95% CI'].values[0]

            # Determine the best single modality
            best_single_mean = max(struct_mean, text_mean)
            best_single_name = "Text-only" if text_mean > struct_mean else "Structured-only"

            # Calculate difference
            diff = comb_mean - best_single_mean

            # Check for statistical significance (two-sided)
            lower_bound_comb = comb_mean - comb_ci
            upper_bound_comb = comb_mean + comb_ci

            if lower_bound_comb > best_single_mean:
                significance = 'Yes (Better)'
            elif upper_bound_comb < best_single_mean:
                significance = 'Yes (Worse)'
            else:
                significance = 'No (Not Significant)'

            comparison_rows.append({
                'Model': model_name,
                'Structured only': f"{struct_mean:.4f}",
                'Text only': f"{text_mean:.4f}",
                'Combined (Mean ± 95% CI)': f"{comb_mean:.4f} ± {comb_ci:.4f}",
                'Δ Combined vs best modality': f"{diff:+.4f}",
                'Meaningfully Different?': significance
            })

    if comparison_rows:
        comparison_df = pd.DataFrame(comparison_rows)
        display(comparison_df)

        print("\nInterpretation:")
        print("The updated significance logic captures both positive and negative impacts. For some models (like Logistic Regression), combining structured features with text actually degrades performance compared to text-only. This indicates that adding structured features introduced noise rather than signal for the linear model.")
    else:
        print("Could not extract complete ablation data for all models.")
else:
    print("Error: 'ablation_df' not found. Please re-run the ablation study.")

## 6.3 Model Comparison (on combined feature matrix)

Cross-validation scores with confidence intervals. Are differences statistically meaningful?

In [ ]:
if 'exploration_df' in locals() and not exploration_df.empty:
    print("--- Model Comparison on Combined Features ---")

    # Sort models by performance
    df_sorted = exploration_df.sort_values(by='Weighted F1 Mean', ascending=False).reset_index(drop=True)

    # Identify the best model to compare against the rest
    best_model_name = df_sorted.loc[0, 'Model']
    best_model_mean = df_sorted.loc[0, 'Weighted F1 Mean']
    best_model_ci = df_sorted.loc[0, 'Weighted F1 95% CI']
    best_lower_bound = best_model_mean - best_model_ci

    comparison_results = []

    for index, row in df_sorted.iterrows():
        model_name = row['Model']
        mean_f1 = row['Weighted F1 Mean']
        ci_f1 = row['Weighted F1 95% CI']
        upper_bound = mean_f1 + ci_f1

        # Determine statistical significance against the best model
        if index == 0:
            significance = "(Best Model)"
        else:
            # More robust check: does the best model's lower bound beat the runner-up's upper bound?
            is_statistically_worse = best_lower_bound > upper_bound
            significance = "Significantly Worse" if is_statistically_worse else "Not Significantly Different"

        comparison_results.append({
            'Model': model_name,
            'Weighted F1 (Mean ± 95% CI)': f"{mean_f1:.4f} ± {ci_f1:.4f}",
            'Macro F1 (Mean ± 95% CI)': f"{row['Macro F1 Mean']:.4f} ± {row['Macro F1 95% CI']:.4f}",
            'vs Best Model': significance
        })

    display(pd.DataFrame(comparison_results))

    print("\nInterpretation:")
    print("We evaluate significance by checking whether the best model's lower CI bound exceeds the comparison model's upper CI bound — a conservative non-overlap test.")

else:
    print("Error: 'exploration_df' not found. Please re-run the Model Exploration.")

## 6.4 Final Model Selection: Selecting the final model and defending it:

**Why this model over the runner-up?**
- While previous part concluded the Voting Classifier was the best out-of-the-box model, we ultimately selected the tuned HistGradientBoostingClassifier as our final model.
- The Voting Classifier achieved a weighted F1 score of 0.60, but it is computationally heavy (running three separate models) and difficult to tune jointly with the text representation.
- By jointly tuning the HistGradientBoostingClassifier alongside the text representation parameters, we achieved a highly competitive weighted F1 score (0.6068) with a single, streamlined pipeline that effectively captures non-linear interactions between text and structured data. The performance difference between the two models (~0.0068) falls within the cross-validation margin of error, meaning we chose the HGB for its operational simplicity rather than statistically significant predictive superiority.

**Is your representation choice portable (would it hold on new data) or overfitted to quirks of your training set?**
- The chosen text representation incorporates TF-IDF with max_features=5000, ngram_range=(1,1), min_df=2, and TruncatedSVD with n_components=200.

- 'ngram_range=(1,1)' (unigrams): The best performance with unigrams suggests that individual words carry the most significant sentiment signal. Single word sentiment (e.g., 'terrible', 'excellent') appears to be highly indicative and less prone to overfitting than overly specific multi-word phrases.

- 'min_df=2': The tuning process selected min_df=2, which optimized cross-validation performance by retaining as much vocabulary as possible from the small (2,000-sample) dataset. However, we must acknowledge a trade-off here: a word appearing in only 2 documents is still extremely rare and likely contains some noise. While this maximized signal on our specific training set, a higher min_df (like 5 or 10) would be a more conservative choice and potentially more portable to new, unseen data to prevent overfitting.

- 'max_features=5000' & 'n_components=200': These relatively high values ensure that a substantial portion of the textual information is retained and reduced to meaningful latent components without excessive loss.

**What trade-offs did you accept?**
- Simplicity vs. Tiny Gains: We accepted a single, slightly less complex pipeline (one tuned HGB model instead of a massive three-model Voting Classifier). While the Voting Classifier might capture some additional ensemble nuance, the tuned HGB delivers comparable, highly competitive performance without the massive computational overhead.
- Recall vs. Precision (Calibration): A 0.50 threshold for predicting ratings 1–2 provides a strong balance, achieving 0.83 recall and 0.92 precision. Lowering the threshold could catch more dissatisfied customers but would increase false positives and operational costs. This choice balances detection and efficiency.
- CV Optimization vs. Strict Portability: As noted above, we accepted the `min_df=2` parameter to maximize CV performance on this small dataset, accepting the risk that it might be slightly overfitted to rare terms compared to a more conservative threshold.

**What do we conceptually expect the TF-IDF top features / SVD components to contribute (based on problem context, since we did not explicitly extract them)?**

- Expected Sentiment Indicators: We hypothesize that TF-IDF highlights important words tied to sentiment. Negative terms (e.g., “broken,” “defective”) would naturally link to low ratings, while positive terms (e.g., “excellent,” “love”) signal high ratings, providing the core classification signal.
- Expected Issue Identification: Specific keywords likely reveal the source of dissatisfaction. Words like “shipping” or “late” would suggest logistics issues, while “quality” or “cheap” point to product problems. Combining this textual context with structured data helps handle misleading reviews where sentiment and rating don’t match.
- Expected SVD Semantic Themes: Conceptually, TruncatedSVD reduces the sparse TF-IDF matrix into key latent themes (e.g., underlying concepts of product quality, delivery, value). With 200 components, we expect the model captures detailed thematic patterns, allowing it to distinguish different, subtle reasons behind a given rating, such as distinguishing delivery-driven dissatisfaction from product quality complaints.

## 6.5 Error Analysis

**Where does the model fail?**
- The model primarily fails on 'deceptive' reviews (10% of data) where text sentiment contradicts the assigned rating, and on subtle distinctions between adjacent ratings (e.g., 1 vs. 2 or 4 vs. 5).

**Are certain classes harder (multi-class)?**
- Yes. Ratings 1 and 2 (dissatisfied) are harder to predict due to lower support in the training set compared to ratings 3 and 4. Rating 5 also shows lower recall as the model tends to play it safe by predicting a 4 for lukewarm positive text.

**Are certain text lengths or structured feature combinations problematic? Can you characterize the failures?**

- Short Text: Reviews under 45 words provide insufficient TF-IDF signal, forcing the model to rely on weak structured correlations.

- Conflicting Signals: High 'delivery_days' combined with highly positive 'review_text' consistently confuses the model, as it must weigh logistical failure against product satisfaction.

## 6.6 Reflection 

### Text Representation Decisions

TF-IDF worked well since this problem contains short reviews where keywords are very important. Words like "broken", "defective", and others indicate that this is a low rating review, while other words like "excellent", "perfect", and more are signals of high ratings. TF-IDF can highlight these words and downweight the more common words that do not give any significant information.

By tuning, TF-IDF parameters have been improved, or the original parameters were updated. For example, ngrma_range was changed from (1,2) to (1,1), since unigrams performed better than bigrams. min_df which is removing rare words parameter was reduced from 5 to 2, since it helps to have more meaningful rare words. Additionally, SVD n_components increased from 100 to 200, because it helps to keep more important vocabulary. These updates helped to make the model perform better.

Using only bigrams or trigrams will reduce the performance since single words sometimes provide stronger expression that bigrams and trigrams may missed, and bigrams and trigrams are rarer, and since this dataset is not large, it will only reduce performance. Additionally, begrams and trigrams can be a reason for weak generalization due to limited by local context. Based on the performance, we can see that unigrams outperformed here bigrams and trigrams, which only proves the above statements.

Word embeddings may be helpful here, but they will not improve a lot. Word embeddings will potentially help with words that are similar and help with context, but since reviews are not very long, the dataset is small, and reviews are pretty clear, TF-IDF is capturing well, and there is no need to use word embeddings. Changing can also be affected by adding complexity and longer training time, and it will be harder to interpret, which is not beneficial in this case.

Some information has been lost in the bag-of-words assumption. It happens because of the context, word order issue, or using words in the wrong way (using "perfect" in a negative way, like sarcasm), or having a good review with a mistakenly low rating. This leads to some performance loss, and we can see that 10% is the cap for deceptive reviews where text sentiment contradicts the actual rating, so we need an additional context, and there are possible some class confusion as well. However, other features, such as delivery_days, seller_rating, or others, can be helpful to adjust and correctly identify dissatisfaction, even with some issues with clarity.

### Structured vs Text Feature Contribution

The results shows that text contributed more to predictive accuracy than structured features. Text-only models achieved weighted F1 scores around 0.54 to 0.58 across all model types, while structured-only models scored only 0.29 to 0.34. This gap is large and consistent, meaning the review text carries the dominant signal for prediciting customer ratings
The reason text dominates is because infromation content. The word choice reflects costumer's sentiment such as "terrible", or "damaged" are strong dissatisfaction that structrured features simply can't replicate (product_price or product_age_months don't indicate whether a customer is happy or not)

However, strcutured features still add value when combined with text, especially for cases where text is misleading. A example of this is the "deceptive review" in this case. Approximately 10% of reviews have text sentiment that doesn't match the real rating. 1 customer might write a neutral or midly postive review about the product but give 1-star because late delivery or poor seller experience. In these cases, structured features like delivery_days and return_initiated provide the additional context that text misses

The way the model processes each modality also matters. Tree-nased models like RF and HistGB can handle both dense structured features and SVD-reduced rexr components in the same feature space, which allow them to learn interaction effects between modalities.

###  Counterfactual Situations

If text were unavailable, preformance would drop significantly from a weighted F1 of approx 0.61 down to around 0.34, based on structured-only results. The drop of 0.27 weighted F1 is large enough to make the model less reliable. It would still be deployable but its business value would be limited. It could still flag some dissatisfied custimers using signals like return_initated, delivery_days, seller_rating but it would miss the majority of cases where dissatisfaction is expressed through language

With 10 times more training data, the component would benefit most is the text representation. Currently, the vocabulary after cleaning is 324 unique words, which is small. A larger dataset would expose the model to much richer and more varied vocabulary, making TF-IDF dearures more informative and allowing bigrams or trigrams to become useful (right now they hurt oerfirmance due to sparsity)

If the text were in a different language, the current pipeline would need significant changes. The TF-IDF stoo words list is set to English so it would need to be replaced with a language-appropriate list. The vocabulary and n-gram behavior would also shift. If the text were much noisier with spelling errors, slang, or inconsistent formatting, we would need additional preprocessing steps such as spell correction or normalization before TF-IDF, since the current pipeline does minimal cleaning beyond lowercasing and punctuation removal. If reviews were much shorter, the bag-of-words representation would become even sparser and less reliable, and this would be the scenario where word embeddings or sentence transformers would provide the most meaningful improvement over TF-IDF, since they can extract semantic meaning from very few words.

The part of the pipeline most sensitive to distributional shift is the TF-IDF vectorizer, because it is fit on training data and its vocabulary and IDF weights are fixed at training time. If the post-deployment review language shifts, for example, customers start using different terminology, or the product catalog expands into new categories with new vocabulary, the TF-IDF representation will become stale and less informative without retraining.